# Parser di documenti per knowledge base

Progetto del corso *Programmazione con Python* — Master ProfessionAI.

Programma Python che, data una cartella di documenti in formati diversi (`.txt`, `.md`, `.csv`), ne estrae contenuto e metadati, li normalizza in una rappresentazione comune e costruisce un indice della knowledge base, salvato in JSON e interrogabile da riga di comando.

**Repository:** [https://github.com/massimiliano-cantore/kb-parser](https://github.com/massimiliano-cantore/kb-parser) — codice, moduli, archivio di prova e README con le istruzioni.

Il notebook è diviso in tre parti:

- **Parte A** — uso del programma da riga di comando sull'archivio di prova;
- **Parte B** — uso delle classi da Python: ricaricamento dell'indice, ricerche, estensione con un nuovo formato;
- **Parte C** — prova di robustezza su file problematici creati al momento, che non fanno parte dell'archivio di prova.

Per eseguirlo: *Runtime → Esegui tutte*.

## Installazione

Clonazione del repository e installazione del pacchetto. Il pacchetto usa solo la libreria standard, quindi non installa dipendenze esterne.

L'ultima riga rende il pacchetto importabile subito anche nelle celle Python: in Colab un'installazione con `pip install -e` diventa visibile al notebook solo dopo un riavvio del runtime.

In [1]:
import os
import sys

if not os.path.exists("/content/kb-parser"):
    !git clone -q https://github.com/massimiliano-cantore/kb-parser.git /content/kb-parser

%cd /content/kb-parser
!pip install -q -e .

if "/content/kb-parser/src" not in sys.path:
    sys.path.insert(0, "/content/kb-parser/src")

# Parte A — Riga di comando

## A.1 L'archivio di prova

Diciotto file fittizi nei tre formati, distribuiti in sottocartelle, con alcuni casi limite costruiti apposta: file vuoto, caratteri speciali, codifiche Latin-1 e Windows-1252, Markdown senza titolo, file molto lungo, estensione non prevista, CSV malformato, file binario con estensione `.txt` e un file nascosto. La descrizione di ogni caso è nel README.

In [2]:
!find sample_archive -type f | sort

sample_archive/.bozza_nascosta.txt
sample_archive/backup_corrotto.txt
sample_archive/file_vuoto.txt
sample_archive/manuale_piattaforma.pdf
sample_archive/note/appunti_riunione_fornitori.md
sample_archive/note/caratteri_speciali.txt
sample_archive/note/idee_nuovi_corsi.txt
sample_archive/procedure/accoglienza_nuovi_docenti.md
sample_archive/procedure/gestione_aule_laboratorio.md
sample_archive/procedure/procedura_backup_piattaforma.txt
sample_archive/registro_presenze_rotto.csv
sample_archive/schede_tecniche/catalogo_corsi_2026.csv
sample_archive/schede_tecniche/fornitori_windows.csv
sample_archive/schede_tecniche/inventario_laboratorio.csv
sample_archive/schede_tecniche/scheda_videoproiettore_aula3.md
sample_archive/storico/storico_interventi_manutenzione.txt
sample_archive/verbali/2025/verbale_comitato_didattico.md
sample_archive/verbali/2025/verbale_riunione_qualita_marzo.txt
sample_archive/verbali/verbale_collegio_docenti_2024.txt


## A.2 Costruzione dell'indice e riepilogo

Il comando `build` legge la cartella e le sottocartelle, applica a ogni file il lettore del suo formato e salva l'indice in `index.json`.

I file che non si possono elaborare vengono segnalati e saltati senza interrompere il processo. Al termine il programma stampa il riepilogo: documenti elaborati, documenti scartati e relative motivazioni.

In [3]:
!python -m kb_parser build sample_archive --output index.json

Archive: sample_archive
Files found: 18
Processed:   14
Skipped:     4
  - corrupted content: 2
  - empty file: 1
  - unsupported format: 1

Skipped files:
  - backup_corrotto.txt [corrupted content] the file contains null bytes, it looks like a binary file
  - file_vuoto.txt [empty file] the file is empty
  - manuale_piattaforma.pdf [unsupported format] extension '.pdf' is not supported (supported: .csv, .markdown, .md, .txt)
  - registro_presenze_rotto.csv [corrupted content] line 3 has 5 fields, expected 4 as in the header

Index saved to index.json (14 documents).


## A.3 L'indice salvato in JSON

Ogni documento è rappresentato con la stessa struttura, qualunque sia il formato di origine: titolo, data (se presente), numero di parole, estratto, formato, percorso e contenuto normalizzato.

In [4]:
!head -n 30 index.json

{
  "format_version": 1,
  "created_at": "2026-11-02T13:47:22",
  "source_folder": "sample_archive",
  "document_count": 14,
  "documents": [
    {
      "title": "appunti_riunione_fornitori",
      "date": null,
      "word_count": 42,
      "excerpt": "Appunti presi durante la telefonata con il fornitore dei banchi didattici. Disponibilità di due banchi aggiuntivi entro fine trimestre Il prezzo resta quello del listino in vigore Chiedere se la…",
      "file_format": "md",
      "path": "note/appunti_riunione_fornitori.md",
      "content": "Appunti presi durante la telefonata con il fornitore dei banchi didattici.\n\nDisponibilità di due banchi aggiuntivi entro fine trimestre\nIl prezzo resta quello del listino in vigore\nChiedere se la garanzia copre anche gli alimentatori\n\nDa ricontattare dopo la riunione di budget."
    },
    {
      "title": "Prova di caratteri speciali – àèéìòù ÀÈÉÌÒÙ ç ñ ß",
      "date": null,
      "word_count": 55,
      "excerpt": "Prova di caratteri sp

## A.4 Elenco dei documenti

Ordinamento predefinito per percorso; con `--sort` anche per titolo, data o lunghezza.

In [5]:
!python -m kb_parser list --index index.json

14 documents in index.json

PATH                                                FORMAT  DATE          WORDS  TITLE
note/appunti_riunione_fornitori.md                  md      -                42  appunti_riunione_fornitori
note/caratteri_speciali.txt                         txt     -                55  Prova di caratteri speciali – àèéìòù ÀÈÉÌÒÙ …
note/idee_nuovi_corsi.txt                           txt     -                44  Idee per nuovi corsi
procedure/accoglienza_nuovi_docenti.md              md      2025-09-01      122  Procedura di accoglienza dei nuovi docenti
procedure/gestione_aule_laboratorio.md              md      2026-01-15       90  Gestione delle aule e dei laboratori
procedure/procedura_backup_piattaforma.txt          txt     2025-11-20       98  Procedura di backup della piattaforma e-lear…
schede_tecniche/catalogo_corsi_2026.csv             csv     -                93  catalogo_corsi_2026
schede_tecniche/fornitori_windows.csv               csv     -                6

In [6]:
!python -m kb_parser list --sort date --index index.json

14 documents in index.json

PATH                                                FORMAT  DATE          WORDS  TITLE
storico/storico_interventi_manutenzione.txt         txt     2021-03-13    55497  Storico interventi di manutenzione 2019-2025
schede_tecniche/scheda_videoproiettore_aula3.md     md      2023-02-10       68  Scheda tecnica: videoproiettore aula 3
verbali/verbale_collegio_docenti_2024.txt           txt     2024-12-20       82  Verbale del collegio docenti
verbali/2025/verbale_riunione_qualita_marzo.txt     txt     2025-03-12       93  VERBALE RIUNIONE DEL GRUPPO QUALITÀ
verbali/2025/verbale_comitato_didattico.md          md      2025-06-18       60  Verbale del comitato didattico
procedure/accoglienza_nuovi_docenti.md              md      2025-09-01      122  Procedura di accoglienza dei nuovi docenti
procedure/procedura_backup_piattaforma.txt          txt     2025-11-20       98  Procedura di backup della piattaforma e-lear…
schede_tecniche/inventario_laboratorio.csv       

## A.5 Ricerca per parola chiave

La ricerca avviene nel titolo e nel contenuto, senza distinzione tra maiuscole e minuscole, e trova anche parti di parola. I risultati sono ordinati per rilevanza (prima i documenti con la parola nel titolo, poi per numero di occorrenze) e mostrano un frammento di testo con la parola tra parentesi quadre.

In [7]:
!python -m kb_parser search "manutenzione" --index index.json

2 documents contain 'manutenzione':

1. Storico interventi di manutenzione 2019-2025
   storico/storico_interventi_manutenzione.txt  (in title, 1 in content)
   Storico interventi di [manutenzione] 2019-2025 Registro completo degli interventi tecnici su at…

2. Scheda tecnica: videoproiettore aula 3
   schede_tecniche/scheda_videoproiettore_aula3.md  (1 in content)
   …ressi: HDMI ×2, VGA, LAN Data di installazione: 2023-02-10 [Manutenzione] Pulire il filtro dell'aria ogni 500 ore di utilizzo. La la…


In [8]:
!python -m kb_parser search "LABORATORIO" --index index.json

7 documents contain 'LABORATORIO':

1. inventario_laboratorio
   schede_tecniche/inventario_laboratorio.csv  (in title)
   codice: LAB-001; descrizione: Banco didattico PLC; quantità: 6; stato: operativo; ultima verifica: 10/01/2026 codice: LAB-002; descrizione: Quadro elettrico didattico; quantità: 2; stato: operativo…

2. Gestione delle aule e dei laboratori
   procedure/gestione_aule_laboratorio.md  (2 in content)
   …otazione Le aule si prenotano dal calendario condiviso. Il [laboratorio] di automazione (aula 3) richiede la presenza di un tecnico…

3. Idee per nuovi corsi
   note/idee_nuovi_corsi.txt  (1 in content)
   …orsi - Corso breve su Git e GitHub per i docenti tecnici - [Laboratorio] di robotica educativa per le scuole medie - Percorso serale…

4. Procedura di accoglienza dei nuovi docenti
   procedure/accoglienza_nuovi_docenti.md  (1 in content)
   …a. Il primo giorno il tutor d'aula accompagna il docente in [laboratorio]. Documenti da consegnare Curriculum aggiornato in f

## A.6 Filtro per formato

In [9]:
!python -m kb_parser filter csv --index index.json

3 documents with format 'csv'

PATH                                                FORMAT  DATE          WORDS  TITLE
schede_tecniche/catalogo_corsi_2026.csv             csv     -                93  catalogo_corsi_2026
schede_tecniche/fornitori_windows.csv               csv     -                61  fornitori_windows
schede_tecniche/inventario_laboratorio.csv          csv     2026-01-10       80  inventario_laboratorio


In [10]:
!python -m kb_parser filter md --index index.json

5 documents with format 'md'

PATH                                                FORMAT  DATE          WORDS  TITLE
note/appunti_riunione_fornitori.md                  md      -                42  appunti_riunione_fornitori
procedure/accoglienza_nuovi_docenti.md              md      2025-09-01      122  Procedura di accoglienza dei nuovi docenti
procedure/gestione_aule_laboratorio.md              md      2026-01-15       90  Gestione delle aule e dei laboratori
schede_tecniche/scheda_videoproiettore_aula3.md     md      2023-02-10       68  Scheda tecnica: videoproiettore aula 3
verbali/2025/verbale_comitato_didattico.md          md      2025-06-18       60  Verbale del comitato didattico


## A.7 Statistiche di sintesi

Documenti per tipo, documenti più lunghi, date e file scartati durante la costruzione.

In [11]:
!python -m kb_parser stats --index index.json

Index: index.json
Built from: sample_archive on 2026-11-02T13:47:22

Documents:        14
Total words:      56485
Average words:    4035
With a date:      9
Without a date:   5
Date range:       2021-03-13 - 2026-01-15

Documents by format:
  txt        6
  md         5
  csv        3

Longest documents (top 5):
     55497 words  storico/storico_interventi_manutenzione.txt
       122 words  procedure/accoglienza_nuovi_docenti.md
        98 words  procedure/procedura_backup_piattaforma.txt
        93 words  schede_tecniche/catalogo_corsi_2026.csv
        93 words  verbali/2025/verbale_riunione_qualita_marzo.txt

Files skipped at build time: 4
  - corrupted content: 2
  - empty file: 1
  - unsupported format: 1


## A.8 Scheda completa di un documento

Il verbale qui sotto è salvato in codifica Latin-1: le lettere accentate vengono lette correttamente.

In [12]:
!python -m kb_parser show verbali/verbale_collegio_docenti_2024.txt --index index.json

Title:   Verbale del collegio docenti
Path:    verbali/verbale_collegio_docenti_2024.txt
Format:  txt
Date:    2024-12-20
Words:   82
Excerpt: Verbale del collegio docenti Data: 20/12/2024 Il collegio si riunisce per l'analisi dell'anno formativo. Partecipano 14 docenti. Si discute della qualità dei materiali didattici: è emersa l'esigenza…

Verbale del collegio docenti

Data: 20/12/2024

Il collegio si riunisce per l'analisi dell'anno formativo. Partecipano 14 docenti.
Si discute della qualità dei materiali didattici: è emersa l'esigenza di uniformare le
dispense, perché oggi ciascun docente usa un modello proprio.
Il coordinatore propone un modello unico; la proposta è approvata all'unanimità.
Si ricorda che l'attività di laboratorio richiede la presenza del tecnico già alle 8.30.
Nota sul file: questo verbale è salvato in codifica Latin-1 (ISO-8859-1), come i documenti
prodotti dal vecchio gestionale.


## A.9 Ricaricamento dell'indice senza rifare il parsing

I comandi `list`, `search`, `filter`, `stats` e `show` leggono soltanto `index.json`. Per dimostrarlo, la cartella dei documenti viene rinominata temporaneamente: la ricerca funziona ugualmente, perché il programma non rilegge i file originali.

In [13]:
!mv sample_archive sample_archive_hidden
!python -m kb_parser search "verbale" --index index.json
!mv sample_archive_hidden sample_archive

3 documents contain 'verbale':

1. Verbale del collegio docenti
   verbali/verbale_collegio_docenti_2024.txt  (in title, 2 in content)
   [Verbale] del collegio docenti Data: 20/12/2024 Il collegio si riun…

2. Verbale del comitato didattico
   verbali/2025/verbale_comitato_didattico.md  (in title, 1 in content)
   [Verbale] del comitato didattico Riunione del 2025-06-18, modalità m…

3. VERBALE RIUNIONE DEL GRUPPO QUALITÀ
   verbali/2025/verbale_riunione_qualita_marzo.txt  (in title, 1 in content)
   [VERBALE] RIUNIONE DEL GRUPPO QUALITÀ Data: 12/03/2025 Luogo: sede d…


## A.10 Errori dell'utente

Un indice inesistente o una cartella sbagliata producono un messaggio chiaro, senza traceback, e un codice di uscita diverso da zero.

In [14]:
!python -m kb_parser stats --index indice_inesistente.json
!python -m kb_parser build cartella_inesistente

Error: indice_inesistente.json: the index file does not exist, run the 'build' command first


Error: cartella_inesistente: the archive folder does not exist


# Parte B — Uso da Python

## B.1 Ricaricamento dell'indice

`KnowledgeBaseIndex.load()` ricostruisce l'indice dal file JSON: ogni scheda torna a essere un oggetto `Document`, senza leggere di nuovo i documenti originali.

In [15]:
from kb_parser.index import KnowledgeBaseIndex

index = KnowledgeBaseIndex.load("index.json")
print(f"Documenti caricati da index.json: {len(index)}")
print(f"Indice creato il {index.created_at} dalla cartella '{index.source_folder}'")

document = index.get("procedure/accoglienza_nuovi_docenti.md")
print()
print(type(document).__name__)
print("title:      ", document.title)
print("date:       ", document.date)
print("word_count: ", document.word_count)
print("file_format:", document.file_format)
print("path:       ", document.path)
print("excerpt:    ", document.excerpt)

Documenti caricati da index.json: 14
Indice creato il 2026-11-02T13:47:22 dalla cartella 'sample_archive'

Document
title:       Procedura di accoglienza dei nuovi docenti
date:        2025-09-01
word_count:  122
file_format: md
path:        procedure/accoglienza_nuovi_docenti.md
excerpt:     Procedura di accoglienza dei nuovi docenti Codice: PRO-DID-04 · Revisione: 3 · Data di emissione: 2025-09-01 Scopo Definire i passaggi con cui Helix Formazione accoglie i docenti al primo incarico…


## B.2 Ricerca, filtro e statistiche

In [16]:
for result in index.search("filtro"):
    print(f"{result.document.title}  (nel titolo: {result.in_title}, occorrenze: {result.occurrences})")
    print(f"    {result.snippet}")

print()
print("Documenti CSV:", [doc.path for doc in index.filter_by_format("csv")])

stats = index.statistics(top=3)
print()
print("Documenti per formato:", stats.documents_by_format)
print("I tre più lunghi:     ", [(doc.path, doc.word_count) for doc in stats.longest])
print("Scartati alla build:  ", stats.skipped_by_reason)

Scheda tecnica: videoproiettore aula 3  (nel titolo: False, occorrenze: 1)
    …Data di installazione: 2023-02-10 Manutenzione Pulire il [filtro] dell'aria ogni 500 ore di utilizzo. La lampada va sostituit…

Documenti CSV: ['schede_tecniche/catalogo_corsi_2026.csv', 'schede_tecniche/fornitori_windows.csv', 'schede_tecniche/inventario_laboratorio.csv']

Documenti per formato: {'txt': 6, 'md': 5, 'csv': 3}
I tre più lunghi:      [('storico/storico_interventi_manutenzione.txt', 55497), ('procedure/accoglienza_nuovi_docenti.md', 122), ('procedure/procedura_backup_piattaforma.txt', 98)]
Scartati alla build:   {'corrupted content': 2, 'empty file': 1, 'unsupported format': 1}


## B.3 Estendere il programma con un nuovo formato

Ogni formato è una classe derivata da `BaseReader`, che contiene i passi comuni (lettura, codifica, controlli, data, estratto, conteggio delle parole). Una nuova classe deve solo dichiarare le estensioni e implementare `parse()`.

Qui si aggiunge il supporto ai file `.log` con una sola classe, senza modificare scanner, indice o interfaccia.

In [17]:
from pathlib import Path

from kb_parser.readers import BaseReader, register_reader, supported_extensions
from kb_parser.scanner import ArchiveScanner


class LogReader(BaseReader):
    """Reads .log files: the first line is the title, the content is the full text."""

    extensions = (".log",)

    def parse(self, text: str, path: Path) -> tuple[str | None, str]:
        first_line = text.strip().split("\n")[0]
        return first_line, text


demo = Path("/content/demo_log")
demo.mkdir(exist_ok=True)
(demo / "server.log").write_text(
    "Log del server della piattaforma\n2026-01-10 08:00 avvio\n2026-01-10 08:05 backup completato\n",
    encoding="utf-8",
)

_, report = ArchiveScanner(demo).scan()
print("Prima della registrazione:")
print(report.summary())

if ".log" not in supported_extensions():
    register_reader(LogReader)

documents, report = ArchiveScanner(demo).scan()
print()
print("Dopo la registrazione:")
print(report.summary())
print(documents[0].title, "|", documents[0].date, "|", documents[0].word_count, "parole")

Prima della registrazione:
Archive: /content/demo_log
Files found: 1
Processed:   0
Skipped:     1
  - unsupported format: 1

Skipped files:
  - server.log [unsupported format] extension '.log' is not supported (supported: .csv, .markdown, .md, .txt)

Dopo la registrazione:
Archive: /content/demo_log
Files found: 1
Processed:   1
Skipped:     0
Log del server della piattaforma | 2026-01-10 | 12 parole


# Parte C — Prova di robustezza

Il parser deve reggere su archivi che non sono stati preparati per lui. La cella seguente crea al momento una cartella con file problematici diversi da quelli dell'archivio di prova, insieme ad alcuni file validi, e la elabora: ogni problema viene segnalato con il suo motivo e l'elaborazione prosegue sugli altri file.

In [18]:
import random
from pathlib import Path

test = Path("/content/robustness_test")
(test / "sottocartella" / "più livelli").mkdir(parents=True, exist_ok=True)
rng = random.Random(0)

files = {
    # validi, ma insoliti
    "NOTE IN MAIUSCOLO.TXT": "Nota con estensione maiuscola e spazi nel nome.".encode("utf-8"),
    "unicode_windows.txt": "Nota salvata dal Blocco note di Windows in UTF-16".encode("utf-16"),
    "sottocartella/più livelli/annotazione è.md": "# Titolo con accenti\nCartelle annidate e nomi accentati.".encode("utf-8"),
    "solo_una_colonna.csv": "matricola\nM-001\nM-002\n".encode("utf-8"),
    # problematici
    "senza_estensione": b"Un file senza estensione",
    "documento.docx": b"PK\x03\x04 finto documento Word",
    "immagine_rinominata.txt": bytes(rng.randrange(256) for _ in range(2048)),
    "solo_spazi.md": b"   \n\t\n   ",
    "solo_simboli.md": b"---\n***\n___\n",
    "virgolette_aperte.csv": 'codice,nota\nA1,"nota senza chiusura\n'.encode("utf-8"),
    "solo_intestazione.csv": b"codice;descrizione\n",
    "colonne_sbagliate.csv": b"a;b;c\n1;2;3\n4;5\n",
}
for name, data in files.items():
    (test / name).write_bytes(data)

!python -m kb_parser build /content/robustness_test --output /content/robustness_index.json

Archive: /content/robustness_test
Files found: 12
Processed:   4
Skipped:     8
  - corrupted content: 3
  - empty file: 3
  - unsupported format: 2

Skipped files:
  - colonne_sbagliate.csv [corrupted content] line 3 has 2 fields, expected 3 as in the header
  - documento.docx [unsupported format] extension '.docx' is not supported (supported: .csv, .markdown, .md, .txt)
  - immagine_rinominata.txt [corrupted content] the file contains null bytes, it looks like a binary file
  - senza_estensione [unsupported format] the file has no extension
  - solo_intestazione.csv [empty file] the CSV file has no data rows
  - solo_simboli.md [empty file] the file contains no textual content
  - solo_spazi.md [empty file] the file contains only whitespace
  - virgolette_aperte.csv [corrupted content] invalid CSV syntax at line 2 (unexpected end of data)

Index saved to /content/robustness_index.json (4 documents).


In [19]:
!python -m kb_parser list --index /content/robustness_index.json

4 documents in /content/robustness_index.json

PATH                                                FORMAT  DATE          WORDS  TITLE
NOTE IN MAIUSCOLO.TXT                               txt     -                 8  Nota con estensione maiuscola e spazi nel no…
solo_una_colonna.csv                                csv     -                 4  solo_una_colonna
sottocartella/più livelli/annotazione è.md          md      -                 8  Titolo con accenti
unicode_windows.txt                                 txt     -                 9  Nota salvata dal Blocco note di Windows in U…
